In [3]:
!pip install pymupdf



In [5]:
import fitz  # PyMuPDF
import os

pdf_folder = "athletic/bibs"
output_folder = "athletic/bibs"
os.makedirs(output_folder, exist_ok=True)

for file in os.listdir(pdf_folder):
    if file.lower().endswith(".pdf"):
        pdf_path = os.path.join(pdf_folder, file)
        doc = fitz.open(pdf_path)
        base_name = os.path.splitext(file)[0]

        for page_number, page in enumerate(doc):
            pix = page.get_pixmap(dpi=300)
            output_path = os.path.join(output_folder, f"{base_name}.png")
            pix.save(output_path)

        print(f"✅ {file} converti ({len(doc)} pages)")


✅ 2022_velo_4S.pdf converti (1 pages)
✅ 2023_cap_20km_Lausanne.pdf converti (1 pages)
✅ 2023_cap_escalade_duc.pdf converti (1 pages)
✅ 2023_cap_vortex.pdf converti (1 pages)
✅ 2023_velo_EDT.pdf converti (1 pages)
✅ 2024_cap_20km_Lausanne.pdf converti (1 pages)
✅ 2024_cap_escalade_elite.pdf converti (1 pages)
✅ 2024_cap_vortex.pdf converti (1 pages)
✅ 2024_ski_Engadin.pdf converti (1 pages)
✅ 2024_tri_Huez.pdf converti (1 pages)
✅ 2024_tri_Rumilly.pdf converti (1 pages)
✅ 2024_tri_Troyes.pdf converti (1 pages)
✅ 2025_cap_gva_marathon.pdf converti (1 pages)
✅ 2025_cap_Saint_Genese.pdf converti (1 pages)
✅ 2025_trail_Gets.pdf converti (1 pages)
✅ 2025_trail_hiver_Gets.pdf converti (1 pages)
✅ 2025_tri_Rumilly.pdf converti (1 pages)
✅ 2025_velo_JPP.pdf converti (1 pages)


In [ ]:
# import cv2 
# img = cv2.imread('projects/images/watchmaking2.jpg')
# cv2.imwrite('projects/images/watchmaking2.png', img, [cv2.IMWRITE_PNG_COMPRESSION, 0])

True

# Script d'Optimisation des Images Athletic

Ce script va compresser et optimiser toutes les images du dossier athletic pour améliorer les performances du site.

In [ ]:
# Installation de Pillow si nécessaire
!pip install Pillow

In [6]:
from PIL import Image
import os
from pathlib import Path

def optimize_image(input_path, output_path, max_width=1920, quality=85):
    """
    Optimise une image en la redimensionnant et la compressant
    
    Args:
        input_path: Chemin de l'image source
        output_path: Chemin de l'image optimisée
        max_width: Largeur maximale (défaut: 1920px)
        quality: Qualité de compression JPEG (défaut: 85)
    """
    try:
        # Ouvrir l'image
        img = Image.open(input_path)
        
        # Obtenir la taille originale
        original_size = os.path.getsize(input_path) / 1024  # en KB
        
        # Redimensionner si l'image est trop grande
        if img.width > max_width:
            ratio = max_width / img.width
            new_height = int(img.height * ratio)
            img = img.resize((max_width, new_height), Image.Resampling.LANCZOS)
        
        # Convertir RGBA en RGB si nécessaire (pour les PNG avec transparence)
        if img.mode == 'RGBA':
            # Créer un fond blanc
            background = Image.new('RGB', img.size, (255, 255, 255))
            background.paste(img, mask=img.split()[3])  # 3 est le canal alpha
            img = background
        elif img.mode != 'RGB':
            img = img.convert('RGB')
        
        # Sauvegarder l'image optimisée
        img.save(output_path, 'JPEG', quality=quality, optimize=True)
        
        # Obtenir la nouvelle taille
        new_size = os.path.getsize(output_path) / 1024  # en KB
        
        # Calculer la réduction
        reduction = ((original_size - new_size) / original_size) * 100
        
        return {
            'success': True,
            'original_size': original_size,
            'new_size': new_size,
            'reduction': reduction
        }
    
    except Exception as e:
        return {
            'success': False,
            'error': str(e)
        }

print("✅ Fonction optimize_image créée")

✅ Fonction optimize_image créée


In [7]:
# Configuration des dossiers
bibs_folder = "athletic/bibs"
images_folder = "athletic/images"
backup_folder = "athletic/backups_originaux"

# Créer le dossier de backup s'il n'existe pas
os.makedirs(backup_folder, exist_ok=True)
os.makedirs(f"{backup_folder}/bibs", exist_ok=True)
os.makedirs(f"{backup_folder}/images", exist_ok=True)

print(f"📁 Dossiers configurés:")
print(f"   - Dossards: {bibs_folder}")
print(f"   - Images: {images_folder}")
print(f"   - Backups: {backup_folder}")

📁 Dossiers configurés:
   - Dossards: athletic/bibs
   - Images: athletic/images
   - Backups: athletic/backups_originaux


In [8]:
# ÉTAPE 1: Optimiser les dossards (bibs) - PRIORITÉ HAUTE
print("🎫 OPTIMISATION DES DOSSARDS")
print("=" * 50)

bibs_stats = {
    'total': 0,
    'optimized': 0,
    'failed': 0,
    'total_original_size': 0,
    'total_new_size': 0
}

for filename in os.listdir(bibs_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        input_path = os.path.join(bibs_folder, filename)
        
        # Créer un backup
        backup_path = os.path.join(f"{backup_folder}/bibs", filename)
        if not os.path.exists(backup_path):
            import shutil
            shutil.copy2(input_path, backup_path)
        
        # Convertir en JPG (changer l'extension)
        base_name = os.path.splitext(filename)[0]
        output_filename = f"{base_name}.jpg"
        output_path = os.path.join(bibs_folder, output_filename)
        
        # Optimiser avec qualité 85 et max width 1200px (les dossards n'ont pas besoin d'être énormes)
        result = optimize_image(input_path, output_path, max_width=1200, quality=85)
        
        bibs_stats['total'] += 1
        
        if result['success']:
            bibs_stats['optimized'] += 1
            bibs_stats['total_original_size'] += result['original_size']
            bibs_stats['total_new_size'] += result['new_size']
            
            print(f"✅ {filename}")
            print(f"   {result['original_size']:.1f} KB → {result['new_size']:.1f} KB (-{result['reduction']:.1f}%)")
            
            # Supprimer l'ancien PNG si on a créé un JPG
            if filename != output_filename and os.path.exists(output_path):
                os.remove(input_path)
                print(f"   🗑️ Ancien PNG supprimé")
        else:
            bibs_stats['failed'] += 1
            print(f"❌ {filename}: {result['error']}")

print("\n" + "=" * 50)
print(f"📊 RÉSUMÉ DOSSARDS:")
print(f"   Total traité: {bibs_stats['total']}")
print(f"   Optimisés: {bibs_stats['optimized']}")
print(f"   Échecs: {bibs_stats['failed']}")
print(f"   Taille originale: {bibs_stats['total_original_size']:.1f} KB ({bibs_stats['total_original_size']/1024:.1f} MB)")
print(f"   Nouvelle taille: {bibs_stats['total_new_size']:.1f} KB ({bibs_stats['total_new_size']/1024:.1f} MB)")
reduction_total = ((bibs_stats['total_original_size'] - bibs_stats['total_new_size']) / bibs_stats['total_original_size'] * 100) if bibs_stats['total_original_size'] > 0 else 0
print(f"   Réduction totale: {reduction_total:.1f}%")

🎫 OPTIMISATION DES DOSSARDS
✅ 2022_velo_4s.png
   5426.6 KB → 132.5 KB (-97.6%)
   🗑️ Ancien PNG supprimé
✅ 2023_cap_20km_Lausanne.png
   5863.0 KB → 125.4 KB (-97.9%)
   🗑️ Ancien PNG supprimé
✅ 2023_cap_escalade_duc.png
   6782.9 KB → 152.0 KB (-97.8%)
   🗑️ Ancien PNG supprimé
✅ 2023_cap_vortex.png
   3663.7 KB → 102.4 KB (-97.2%)
   🗑️ Ancien PNG supprimé
✅ 2023_velo_EDT.png
   3458.3 KB → 93.0 KB (-97.3%)
   🗑️ Ancien PNG supprimé
✅ 2024_cap_20km_Lausanne.png
   8718.8 KB → 170.4 KB (-98.0%)
   🗑️ Ancien PNG supprimé
✅ 2024_cap_escalade_elite.png
   4311.6 KB → 110.8 KB (-97.4%)
   🗑️ Ancien PNG supprimé
✅ 2024_cap_vortex.png
   11141.0 KB → 158.7 KB (-98.6%)
   🗑️ Ancien PNG supprimé
✅ 2024_ski_Engadin.png
   12374.6 KB → 224.2 KB (-98.2%)
   🗑️ Ancien PNG supprimé
✅ 2024_tri_Huez.png
   7222.7 KB → 144.0 KB (-98.0%)
   🗑️ Ancien PNG supprimé
✅ 2024_tri_Rumilly.png
   7800.0 KB → 155.1 KB (-98.0%)
   🗑️ Ancien PNG supprimé
✅ 2024_tri_Troyes.png
   3754.7 KB → 85.7 KB (-97.7%)
   

In [9]:
# ÉTAPE 2: Optimiser les images principales
print("\n🖼️ OPTIMISATION DES IMAGES PRINCIPALES")
print("=" * 50)

images_stats = {
    'total': 0,
    'optimized': 0,
    'failed': 0,
    'total_original_size': 0,
    'total_new_size': 0
}

for filename in os.listdir(images_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        input_path = os.path.join(images_folder, filename)
        
        # Créer un backup
        backup_path = os.path.join(f"{backup_folder}/images", filename)
        if not os.path.exists(backup_path):
            import shutil
            shutil.copy2(input_path, backup_path)
        
        # Convertir en JPG
        base_name = os.path.splitext(filename)[0]
        output_filename = f"{base_name}.jpg"
        output_path = os.path.join(images_folder, output_filename)
        
        # Optimiser avec qualité 85 et max width 1920px
        result = optimize_image(input_path, output_path, max_width=1920, quality=85)
        
        images_stats['total'] += 1
        
        if result['success']:
            images_stats['optimized'] += 1
            images_stats['total_original_size'] += result['original_size']
            images_stats['total_new_size'] += result['new_size']
            
            print(f"✅ {filename}")
            print(f"   {result['original_size']:.1f} KB → {result['new_size']:.1f} KB (-{result['reduction']:.1f}%)")
            
            # Supprimer l'ancien si différent format
            if filename != output_filename and os.path.exists(output_path):
                os.remove(input_path)
                print(f"   🗑️ Ancien fichier supprimé")
        else:
            images_stats['failed'] += 1
            print(f"❌ {filename}: {result['error']}")

print("\n" + "=" * 50)
print(f"📊 RÉSUMÉ IMAGES PRINCIPALES:")
print(f"   Total traité: {images_stats['total']}")
print(f"   Optimisés: {images_stats['optimized']}")
print(f"   Échecs: {images_stats['failed']}")
print(f"   Taille originale: {images_stats['total_original_size']:.1f} KB ({images_stats['total_original_size']/1024:.1f} MB)")
print(f"   Nouvelle taille: {images_stats['total_new_size']:.1f} KB ({images_stats['total_new_size']/1024:.1f} MB)")
reduction_images = ((images_stats['total_original_size'] - images_stats['total_new_size']) / images_stats['total_original_size'] * 100) if images_stats['total_original_size'] > 0 else 0
print(f"   Réduction totale: {reduction_images:.1f}%")


🖼️ OPTIMISATION DES IMAGES PRINCIPALES
✅ 20km_Lausanne.png
   249.9 KB → 80.2 KB (-67.9%)
   🗑️ Ancien fichier supprimé
✅ 20km_Lausanne_2024.jpg
   338.5 KB → 474.2 KB (--40.1%)
✅ A_venir.png
   11.5 KB → 14.6 KB (--27.2%)
   🗑️ Ancien fichier supprimé
✅ Coureur_bois.jpg
   7.7 KB → 10.0 KB (--29.4%)
✅ Coureur_bois.png
   12.9 KB → 9.8 KB (-24.4%)
   🗑️ Ancien fichier supprimé
✅ Course_du_duc.jpg
   657.1 KB → 703.3 KB (--7.0%)
✅ Course_escalade.png
   539.9 KB → 135.5 KB (-74.9%)
   🗑️ Ancien fichier supprimé
✅ EDT.png
   1143.9 KB → 188.2 KB (-83.5%)
   🗑️ Ancien fichier supprimé
✅ Engadin.jpg
   698.5 KB → 427.2 KB (-38.8%)
✅ GTJ.jpg
   309.2 KB → 322.0 KB (--4.1%)
✅ JPP.png
   58.8 KB → 34.2 KB (-41.8%)
   🗑️ Ancien fichier supprimé
✅ Les_Gets.jpg
   2244.0 KB → 848.8 KB (-62.2%)
✅ Les_Gets_hiver.jpg
   11.1 KB → 14.6 KB (--31.7%)
✅ Marathon_Geneve.jpg
   284.7 KB → 291.7 KB (--2.4%)
✅ Octobre_Rose.jpg
   26.3 KB → 33.6 KB (--27.6%)
✅ Rhode_saint_genese.png
   23.2 KB → 15.0 KB (-

In [11]:
# ÉTAPE 2b: Optimiser les images de la page PROJECTS
print("\n📂 OPTIMISATION DES IMAGES PROJECTS")
print("=" * 50)

projects_folder = "projects/images"
projects_stats = {
    'total': 0,
    'optimized': 0,
    'failed': 0,
    'total_original_size': 0,
    'total_new_size': 0
}

# Créer le dossier de backup pour projects
os.makedirs(f"{backup_folder}/projects_images", exist_ok=True)

for filename in os.listdir(projects_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        input_path = os.path.join(projects_folder, filename)
        
        # Créer un backup
        backup_path = os.path.join(f"{backup_folder}/projects_images", filename)
        if not os.path.exists(backup_path):
            import shutil
            shutil.copy2(input_path, backup_path)
        
        # Convertir en JPG
        base_name = os.path.splitext(filename)[0]
        output_filename = f"{base_name}.jpg"
        output_path = os.path.join(projects_folder, output_filename)
        
        # Optimiser avec qualité 85 et max width 1920px
        result = optimize_image(input_path, output_path, max_width=1920, quality=85)
        
        projects_stats['total'] += 1
        
        if result['success']:
            projects_stats['optimized'] += 1
            projects_stats['total_original_size'] += result['original_size']
            projects_stats['total_new_size'] += result['new_size']
            
            print(f"✅ {filename}")
            print(f"   {result['original_size']:.1f} KB → {result['new_size']:.1f} KB (-{result['reduction']:.1f}%)")
            
            # Supprimer l'ancien si différent format
            if filename != output_filename and os.path.exists(output_path):
                os.remove(input_path)
                print(f"   🗑️ Ancien fichier supprimé")
        else:
            projects_stats['failed'] += 1
            print(f"❌ {filename}: {result['error']}")

print("\n" + "=" * 50)
print(f"📊 RÉSUMÉ IMAGES PROJECTS:")
print(f"   Total traité: {projects_stats['total']}")
print(f"   Optimisés: {projects_stats['optimized']}")
print(f"   Échecs: {projects_stats['failed']}")
print(f"   Taille originale: {projects_stats['total_original_size']:.1f} KB ({projects_stats['total_original_size']/1024:.1f} MB)")
print(f"   Nouvelle taille: {projects_stats['total_new_size']:.1f} KB ({projects_stats['total_new_size']/1024:.1f} MB)")
reduction_projects = ((projects_stats['total_original_size'] - projects_stats['total_new_size']) / projects_stats['total_original_size'] * 100) if projects_stats['total_original_size'] > 0 else 0
print(f"   Réduction totale: {reduction_projects:.1f}%")


📂 OPTIMISATION DES IMAGES PROJECTS
✅ archipelago.png
   467.9 KB → 101.5 KB (-78.3%)
   🗑️ Ancien fichier supprimé
✅ coin.png
   70.7 KB → 62.8 KB (-11.1%)
   🗑️ Ancien fichier supprimé
✅ coopnav.png
   36.9 KB → 20.9 KB (-43.3%)
   🗑️ Ancien fichier supprimé
✅ epflrt.png
   4752.5 KB → 294.4 KB (-93.8%)
   🗑️ Ancien fichier supprimé
✅ legged.png
   507.1 KB → 61.1 KB (-87.9%)
   🗑️ Ancien fichier supprimé
✅ liseagle.png
   31821.7 KB → 385.8 KB (-98.8%)
   🗑️ Ancien fichier supprimé
✅ micro.png
   2244.9 KB → 212.5 KB (-90.5%)
   🗑️ Ancien fichier supprimé
✅ microgrid.png
   364.1 KB → 113.5 KB (-68.8%)
   🗑️ Ancien fichier supprimé
✅ semoir.png
   338.2 KB → 220.5 KB (-34.8%)
   🗑️ Ancien fichier supprimé
✅ silex.png
   1356.8 KB → 272.1 KB (-79.9%)
   🗑️ Ancien fichier supprimé
✅ syslog.png
   689.6 KB → 123.1 KB (-82.2%)
   🗑️ Ancien fichier supprimé
✅ thymio.png
   1181.9 KB → 88.3 KB (-92.5%)
   🗑️ Ancien fichier supprimé
✅ unitraj.png
   46.1 KB → 24.5 KB (-46.8%)
   🗑️ Ancien 

In [ ]:
# ÉTAPE 3: Résumé final et statistiques globales
print("\n" + "=" * 70)
print("🎉 OPTIMISATION TERMINÉE - RÉSUMÉ GLOBAL")
print("=" * 70)

total_files = bibs_stats['total'] + images_stats['total'] + projects_stats['total']
total_optimized = bibs_stats['optimized'] + images_stats['optimized'] + projects_stats['optimized']
total_failed = bibs_stats['failed'] + images_stats['failed'] + projects_stats['failed']

total_original = bibs_stats['total_original_size'] + images_stats['total_original_size'] + projects_stats['total_original_size']
total_new = bibs_stats['total_new_size'] + images_stats['total_new_size'] + projects_stats['total_new_size']
total_saved = total_original - total_new
total_reduction = (total_saved / total_original * 100) if total_original > 0 else 0

print(f"\n📁 Fichiers traités: {total_files}")
print(f"   ✅ Optimisés avec succès: {total_optimized}")
print(f"   ❌ Échecs: {total_failed}")

print(f"\n💾 Espace:")
print(f"   Taille originale totale: {total_original/1024:.2f} MB")
print(f"   Taille optimisée totale: {total_new/1024:.2f} MB")
print(f"   Espace économisé: {total_saved/1024:.2f} MB")
print(f"   Réduction: {total_reduction:.1f}%")

print(f"\n🎯 Impact sur la performance:")
if total_saved > 100000:  # Plus de 100 MB économisés
    print(f"   🚀 Amélioration MAJEURE attendue!")
    print(f"   Le temps de chargement devrait être considérablement réduit.")
elif total_saved > 50000:  # 50-100 MB
    print(f"   ✨ Amélioration SIGNIFICATIVE attendue!")
    print(f"   Le site devrait charger beaucoup plus rapidement.")
else:
    print(f"   ✓ Amélioration modérée attendue.")

print(f"\n📝 Prochaines étapes:")
print(f"   1. Les fichiers originaux sont sauvegardés dans: {backup_folder}")
print(f"   2. Si vous avez converti PNG → JPG, mettez à jour performances.json")
print(f"   3. Testez le site pour vérifier que tout s'affiche correctement")
print(f"   4. Mesurez l'amélioration du temps de chargement")

print("\n" + "=" * 70)


🎉 OPTIMISATION TERMINÉE - RÉSUMÉ GLOBAL

📁 Fichiers traités: 39
   ✅ Optimisés avec succès: 39
   ❌ Échecs: 0

💾 Espace:
   Taille originale totale: 138.74 MB
   Taille optimisée totale: 7.13 MB
   Espace économisé: 131.61 MB
   Réduction: 94.9%

🎯 Impact sur la performance:
   🚀 Amélioration MAJEURE attendue!
   Le temps de chargement devrait être considérablement réduit.

📝 Prochaines étapes:
   1. Les fichiers originaux sont sauvegardés dans: athletic/backups_originaux
   2. Si vous avez converti PNG → JPG, mettez à jour performances.json
   3. Testez le site pour vérifier que tout s'affiche correctement
   4. Mesurez l'amélioration du temps de chargement

